# M2: CLV 조건부 4구간 경제표현 (Dunnhumby, seed 42)

LightGCN ID 64차원에 학습구간 가격 4구간의 사용자 지출구성과 상품 경제위치를 나타내는 4차원 블록을 추가합니다. 사용자 블록은 historical CLV proxy `q_C`로 조건화하고, 희소 이력은 `n/(n+10)`으로 축소합니다. `rho=0.05`는 고정하며 4개 가격구간의 상대가중치만 하나의 BPR 손실에서 공동 학습합니다. 동일 초기화 `rho=0`, 공동학습 ID-only, degree-matched CLV 순열 대조군을 함께 비교하며 최종 test와 holdout은 생성하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
SOURCE_COMMIT = '3e98a9033486c0921c7435a452981fd4dd44d522'
REPO_DIR = '/content/clv-m2-lightgcn-runner'
if len(SOURCE_COMMIT) != 40:
    raise RuntimeError('검토된 소스 커밋을 SOURCE_COMMIT에 고정해야 합니다')
!if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch origin; else git clone {REPO_URL} {REPO_DIR}; fi
!git -C {REPO_DIR} checkout {SOURCE_COMMIT}
%cd {REPO_DIR}


In [ ]:
import json
import torch
from lightgcn_clv_economic_quartile_distribution import (
    configure_economic_quartile_run,
    preflight_summary,
    run_economic_quartile_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_economic_quartile_run()
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))


In [ ]:
result_df = run_economic_quartile_screen(cfg)


In [ ]:
from IPython.display import display
import pandas as pd

print('1) 절대지표: M1, rho=0, 실제 CLV, degree-matched shuffle, ID-only')
display(result_df)
print('2) 대조군별 전체·CLV 구간 성과')
display(pd.DataFrame(result_df.attrs['comparison']))
print('3) rho=0 대비 CLV 구간별 Top-10 변경')
display(pd.DataFrame(result_df.attrs['top10_overlap']))
print('4) 실제 CLV 대 degree-matched shuffle Top-10 변경')
display(pd.DataFrame(result_df.attrs['attribution_overlap']))
print('5) 4구간 경제입력 진단')
print(json.dumps(result_df.attrs['economic_input_diagnostics'], ensure_ascii=False, indent=2))
print('6) 경제블록의 실제 점수 영향력')
display(pd.DataFrame(result_df.attrs['score_diagnostics']))
print('7) 사전 판정 규칙 결과')
print(json.dumps(result_df.attrs['screening_reading'], ensure_ascii=False, indent=2))
print('8) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
